# LoRA 저랭크 미세조정 실습 - Low-Rank Delta W

- Tutorial ID: `mod-lora-lab`
- Tutorial: LoRA 저랭크 미세조정 실습
- Section ID: `lora-1`
- Section: Low-Rank Delta W


In [ ]:
# ============================================================
# 코드 읽는 법 — Low-Rank Delta W
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 큰 가중치 W를 직접 바꾸지 않고 저랭크 A/B 업데이트가 어떻게 더해지는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.random.seed(2)
d_in, d_out, r, alpha = 16, 12, 3, 6
W = np.random.randn(d_out, d_in) / np.sqrt(d_in)
A = np.random.randn(r, d_in) * 0.01
B = np.zeros((d_out, r))  # 초기 delta=0 안정화
x = np.random.randn(d_in)
base = W @ x
lora0 = (alpha/r) * (B @ (A @ x))
print('initial delta norm:', np.linalg.norm(lora0))
# 한 번의 가상 업데이트 뒤
B += np.random.randn(d_out, r) * 0.05
adapted = W @ x + (alpha/r) * (B @ (A @ x))
print('full params:', W.size)
print('LoRA trainable params:', A.size + B.size)
print('compression ratio:', round(W.size/(A.size+B.size), 2), 'x')
print('output changed by:', round(np.linalg.norm(adapted-base), 5))